In [1]:
import os
import yaml
import copy
import time
import numpy as np
import pandas as pd
import xarray as xr

from xclim.indices import (
    standardized_precipitation_evapotranspiration_index,
    water_budget,
)

from xclim.indices.stats import (
    standardized_index_fit_params,
    standardized_index
)

In [1]:
# process var for SPEI workflow
def process_vars(data):
    """
    Preprocess data variables for SPEI calculation.
    """
    data["time"] = pd.to_datetime(data["time"]).to_numpy().astype("datetime64[ns]")
    first_date = data.time.min().dt.strftime("%Y-%m-%d").values.item()

    if not first_date.endswith("-01-01"):
        first_year = int(first_date[:4])
        data = data.sel(time=slice(str(first_year + 1), None))

    precip = data["precip"]
    tmin = data["tmin"]
    tmax = data["tmax"]

    return precip, tmin, tmax


# ============================================================== #
# SPEI config
# -------------------------------------------------------------- #
dist = "fisk"
method = "ML"
# -------------------------------------------------------------- #
# Calibration period: 1968-01-01 to 1999-12-31
# Application period: 1958-01-01 to 2025-12-31 (full record)
# -------------------------------------------------------------- #
cal_start = "1968-01-01"
cal_end   = "1999-12-31"


# ============================================================== #
# Fairbanks
#   - T2_1h_max  / T2_1h_max_ensmean
#   - T2_1d_max  / T2_1d_max_ensmean
#   - T2_7d_max  / T2_7d_max_ensmean
#   - T2_30d_max / T2_30d_max_ensmean
# (ensmean = metric computed on the ensemble-mean field)
# ============================================================== #

stn = 'Fairbanks'
vars_keep = ["TREFHT", "TREFHTMX"]
years = range(1958, 2020)

ds_collection = []

for year in years:
    fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM_SMYLE_STN_MEMBER/{stn}_{year}.zarr'
    ds = xr.open_zarr(fn, consolidated=True)[vars_keep].chunk({"time": -1})
    ds = ds.sel(time=slice(f'{year+1}-01-01', f'{year+10}-12-31'))

    # ---- per-member metrics ----
    da_TREFHTMX_max = ds["TREFHTMX"].groupby("time.year").max("time", skipna=True).rename("T2_1h_max")
    da_TREFHT_max   = ds["TREFHT"].groupby("time.year").max("time", skipna=True).rename("T2_1d_max")

    da_TREFHT_7d_max = (
        ds["TREFHT"]
        .rolling(time=7, min_periods=7).mean()
        .groupby("time.year").max("time", skipna=True)
        .rename("T2_7d_max")
    )

    da_TREFHT_30d_max = (
        ds["TREFHT"]
        .rolling(time=30, min_periods=30).mean()
        .groupby("time.year").max("time", skipna=True)
        .rename("T2_30d_max")
    )

    # ---- ensemble-mean inputs, then metrics ----
    ds_em = ds.mean(dim="member")

    da_TREFHTMX_max_em = ds_em["TREFHTMX"].groupby("time.year").max("time", skipna=True).rename("T2_1h_max_ensmean")
    da_TREFHT_max_em   = ds_em["TREFHT"].groupby("time.year").max("time", skipna=True).rename("T2_1d_max_ensmean")

    da_TREFHT_7d_max_em = (
        ds_em["TREFHT"]
        .rolling(time=7, min_periods=7).mean()
        .groupby("time.year").max("time", skipna=True)
        .rename("T2_7d_max_ensmean")
    )

    da_TREFHT_30d_max_em = (
        ds_em["TREFHT"]
        .rolling(time=30, min_periods=30).mean()
        .groupby("time.year").max("time", skipna=True)
        .rename("T2_30d_max_ensmean")
    )

    ds_merge = xr.merge([
        da_TREFHTMX_max, da_TREFHT_max, da_TREFHT_7d_max, da_TREFHT_30d_max,
        da_TREFHTMX_max_em, da_TREFHT_max_em, da_TREFHT_7d_max_em, da_TREFHT_30d_max_em,
    ])
    ds_merge = ds_merge.assign_coords(year=np.arange(10, dtype=int))
    ds_merge = ds_merge.rename({'year': 'lead_year'})

    ds_collection.append(ds_merge)

ds_all = xr.concat(ds_collection, dim="init_year")
ds_all = ds_all.assign_coords(init_year=np.arange(1959, 2021, dtype=int))

# Chunking: 'member' is silently ignored for ensmean vars that don't have it
ds_all = ds_all.chunk({"init_year": -1, "lead_year": -1, "member": -1})
ds_all = ds_all.transpose('init_year', 'lead_year', 'member', missing_dims='ignore')

ds_all["valid_year"] = ds_all["init_year"] + ds_all["lead_year"]

dir_stn = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{stn}/'
save_name = dir_stn + "CESM_STN.zarr"
ds_all.to_zarr(save_name, mode="w")
print(save_name)

# ============================================================== #
# Pituffik
#   - Melting Degree Days (MDD)         / MDD_ensmean
#   - Freeze-Thaw Days   (FT_days)      / FT_days_ensmean
# (ensmean = metric computed on the ensemble-mean field)
# ============================================================== #

stn = 'Pituffik'
vars_keep = ["TREFHT", "TREFHTMX"]
years = range(1958, 2020)

list_MDD_all = []
thres = 273.15

for year in years:
    fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM_SMYLE_STN_MEMBER/{stn}_{year}.zarr'
    ds = xr.open_zarr(fn, consolidated=True)[vars_keep].chunk({"time": -1})
    ds = ds.sel(time=slice(f'{year+1}-01-01', f'{year+10}-12-31'))

    ds_em = ds.mean(dim="member")

    list_MDD_lead = []
    for lead_year in range(year+1, year+10+1):
        ds_    = ds.sel(time=slice(f"{lead_year}-03-01", f"{lead_year}-06-30"))
        ds_em_ = ds_em.sel(time=slice(f"{lead_year}-03-01", f"{lead_year}-06-30"))

        mdd_mj    = (ds_["TREFHT"]    - thres).clip(min=0).sum(dim="time")
        mdd_mj_em = (ds_em_["TREFHT"] - thres).clip(min=0).sum(dim="time")

        ds_MDD = xr.merge([
            mdd_mj.to_dataset(name="MDD"),
            mdd_mj_em.to_dataset(name="MDD_ensmean"),
        ]).assign_coords(year=lead_year).expand_dims("year")
        list_MDD_lead.append(ds_MDD)

    ds_MDD_lead = xr.merge(list_MDD_lead)
    ds_MDD_lead = ds_MDD_lead.assign_coords(year=np.arange(10))
    list_MDD_all.append(ds_MDD_lead)

ds_MDD_all = xr.concat(list_MDD_all, dim='init_year')
ds_MDD_all = ds_MDD_all.rename({'year': 'lead_year'})
ds_MDD_all = ds_MDD_all.assign_coords({'init_year': np.arange(1958+1, 2020+1)})

ds_MDD_all = ds_MDD_all.chunk({"init_year": -1, "lead_year": -1, "member": -1})
ds_MDD_all = ds_MDD_all.transpose('init_year', 'lead_year', 'member', missing_dims='ignore')

list_FT_all = []
Tf = 273.15  # K
vars_keep = ["TREFHT", "TREFHTMX", 'TREFHTMN']

for year in range(1958, 2020):
    fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM_SMYLE_STN_MEMBER/{stn}_{year}.zarr'
    ds = xr.open_zarr(fn, consolidated=True)[vars_keep].chunk({"time": -1})
    ds = ds.sel(time=slice(f'{year+1}-01-01', f'{year+10}-12-31'))

    ds_em = ds.mean(dim="member")

    list_FT_lead = []
    for lead_year in range(year+1, year+10+1):
        ds_    = ds.sel(time=slice(f"{lead_year}-03-01", f"{lead_year}-06-30"))
        ds_em_ = ds_em.sel(time=slice(f"{lead_year}-03-01", f"{lead_year}-06-30"))

        ft_day    = ((ds_["TREFHTMN"]    < Tf) & (ds_["TREFHTMX"]    > Tf)).astype("int8")
        ft_day_em = ((ds_em_["TREFHTMN"] < Tf) & (ds_em_["TREFHTMX"] > Tf)).astype("int8")

        ft_count    = ft_day.sum(dim="time")
        ft_count_em = ft_day_em.sum(dim="time")

        ds_FT = xr.merge([
            ft_count.to_dataset(name="FT_days"),
            ft_count_em.to_dataset(name="FT_days_ensmean"),
        ]).assign_coords(year=lead_year).expand_dims("year")
        list_FT_lead.append(ds_FT)

    ds_FT_lead = xr.merge(list_FT_lead)
    ds_FT_lead = ds_FT_lead.assign_coords(year=np.arange(10))
    list_FT_all.append(ds_FT_lead)

ds_FT_all = xr.concat(list_FT_all, dim='init_year')
ds_FT_all = ds_FT_all.rename({'year': 'lead_year'})
ds_FT_all = ds_FT_all.assign_coords({'init_year': np.arange(1958+1, 2020+1)})
ds_FT_all = ds_FT_all.chunk({"init_year": -1, "lead_year": -1, "member": -1})
ds_FT_all = ds_FT_all.transpose('init_year', 'lead_year', 'member', missing_dims='ignore')

ds_all = xr.merge([ds_FT_all, ds_MDD_all])
ds_all["valid_year"] = ds_all["init_year"] + ds_all["lead_year"]

dir_stn = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{stn}/'
save_name = dir_stn + "CESM_STN.zarr"
ds_all.to_zarr(save_name, mode="w")
print(save_name)


# ============================================================== #
# Guam
#  - SPEI
# ============================================================== #

stn = 'Guam'
SPEI_03 = np.empty((10, 12*(2020-1959+1), 20))
SPEI_03[...] = np.nan

SPEI_48 = np.empty((10, 12*(2020-1959+1), 20))
SPEI_48[...] = np.nan

SPEI_03_mean = np.empty((10, 12*(2020-1959+1),))
SPEI_03_mean[...] = np.nan

SPEI_48_mean = np.empty((10, 12*(2020-1959+1),))
SPEI_48_mean[...] = np.nan

for lead_year in range(0, 10):
    print(f'process lead year: {lead_year}')
    # ------------------------------------------------------------ #
    # derive param from ensemble mean
    # ------------------------------------------------------------ #
    start_date = f"{1959+lead_year}-01-01T00"
    end_date = f"{2020+lead_year}-12-31T23"

    ds_collection = []

    for year_init in range(1958, 2020, 1):

        year_start = year_init + 1 + lead_year
        time_start = f'{year_start}-01-01T00'
        time_end = f'{year_start}-12-31T00'

        fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM_SMYLE_STN_MEMBER/{stn}_{year_init}.zarr'
        ds_CESM = xr.open_zarr(fn)[['TREFHTMN', 'TREFHTMX', 'PRECT']].mean(['member',])
        ds_CESM = ds_CESM.sel(time=slice(time_start, time_end))
        ds_collection.append(ds_CESM)

    ds_all = xr.concat(ds_collection, dim='time')
    ds_all = ds_all.load()

    cft = ds_all.indexes['time']
    pd_idx = cft.to_datetimeindex(unsafe=True)
    ds_all = ds_all.assign_coords({'time': pd_idx})

    lat_ref = ds_all['lat'].values
    lat_mid = lat_ref
    time_vals = ds_all['time']

    tmin = ds_all['TREFHTMN'].values
    tmax = ds_all['TREFHTMX'].values
    precip = ds_all['PRECT'].values

    ds = xr.Dataset(
        {
            "precip": (("time",), precip*1e3, {"units": "kg m-2 s-1"}),
            "tmin":   (("time",), tmin-273.15, {"units": "degC"}),
            "tmax":   (("time",), tmax-273.15, {"units": "degC"}),
        },
        coords={"time": time_vals, "lat": lat_mid}
    )

    for v in ("precip", "tmin", "tmax"):
        ds[v] = ds[v].assign_coords(lat=lat_mid)
        ds[v]["lat"].attrs = {
            "standard_name": "latitude",
            "units": "degrees_north", "axis": "Y"
        }

    precip, tmin, tmax = process_vars(ds)

    wb = water_budget(
        pr=precip, tasmin=tmin, tasmax=tmax,
        method="HG85", lat=precip["lat"],
    )
    wb.attrs["units"] = "kg m-2 s-1"

    params_03 = standardized_index_fit_params(
        wb.sel(time=slice(cal_start, cal_end)),
        freq="MS",
        window=3,
        dist=dist,
        method=method,
    )

    params_48 = standardized_index_fit_params(
        wb.sel(time=slice(cal_start, cal_end)),
        freq="MS",
        window=48,
        dist=dist,
        method=method,
    )

    spei_03 = standardized_index(
        wb, freq="MS", window=3, dist=dist, method=method,
        params=params_03, zero_inflated=False, fitkwargs=None,
        cal_start=None, cal_end=None,
    )

    spei_48 = standardized_index(
        wb, freq="MS", window=48, dist=dist, method=method,
        params=params_48, zero_inflated=False, fitkwargs=None,
        cal_start=None, cal_end=None,
    )

    SPEI_03_mean[lead_year, :] = spei_03.values
    SPEI_48_mean[lead_year, :] = spei_48.values

    # ------------------------------------------------------------ #
    # SPEI on members
    # ------------------------------------------------------------ #

    for member in range(20):
        start_date = f"{1959+lead_year}-01-01T00"
        end_date = f"{2020+lead_year}-12-31T23"

        ds_collection = []

        for year_init in range(1958, 2020, 1):

            year_start = year_init + 1 + lead_year
            time_start = f'{year_start}-01-01T00'
            time_end = f'{year_start}-12-31T00'

            fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM_SMYLE_STN_MEMBER/{stn}_{year_init}.zarr'
            ds_CESM = xr.open_zarr(fn)[['TREFHTMN', 'TREFHTMX', 'PRECT']].isel(member=member)
            ds_CESM = ds_CESM.sel(time=slice(time_start, time_end))
            ds_collection.append(ds_CESM)

        ds_all = xr.concat(ds_collection, dim='time')
        ds_all = ds_all.load()

        cft = ds_all.indexes['time']
        pd_idx = cft.to_datetimeindex(unsafe=True)
        ds_all = ds_all.assign_coords({'time': pd_idx})

        lat_ref = ds_all['lat'].values
        lat_mid = lat_ref
        time_vals = ds_all['time']

        tmin = ds_all['TREFHTMN'].values
        tmax = ds_all['TREFHTMX'].values
        precip = ds_all['PRECT'].values

        ds = xr.Dataset(
            {
                "precip": (("time",), precip*1e3, {"units": "kg m-2 s-1"}),
                "tmin":   (("time",), tmin-273.15, {"units": "degC"}),
                "tmax":   (("time",), tmax-273.15, {"units": "degC"}),
            },
            coords={"time": time_vals, "lat": lat_mid}
        )

        for v in ("precip", "tmin", "tmax"):
            ds[v] = ds[v].assign_coords(lat=lat_mid)
            ds[v]["lat"].attrs = {
                "standard_name": "latitude",
                "units": "degrees_north", "axis": "Y"
            }

        precip, tmin, tmax = process_vars(ds)

        wb = water_budget(
            pr=precip, tasmin=tmin, tasmax=tmax,
            method="HG85", lat=precip["lat"],
        )
        wb.attrs["units"] = "kg m-2 s-1"

        spei_03 = standardized_index(
            wb, freq="MS", window=3, dist=dist, method=method,
            params=params_03, zero_inflated=False, fitkwargs=None,
            cal_start=None, cal_end=None,
        )

        spei_48 = standardized_index(
            wb, freq="MS", window=48, dist=dist, method=method,
            params=params_48, zero_inflated=False, fitkwargs=None,
            cal_start=None, cal_end=None,
        )

        SPEI_03[lead_year, :, member] = spei_03.values
        SPEI_48[lead_year, :, member] = spei_48.values

# ======================================= #
# numpy to ds
n_lead = 10
n_init = 62
m_per_year = 12

# (10, 744, 20) -> (10, 62, 12, 20)
tmp_03 = SPEI_03.reshape(n_lead, n_init, m_per_year, *SPEI_03.shape[2:])
tmp_48 = SPEI_48.reshape(n_lead, n_init, m_per_year, *SPEI_48.shape[2:])

tmp_03_mean = SPEI_03_mean.reshape(n_lead, n_init, m_per_year)
tmp_48_mean = SPEI_48_mean.reshape(n_lead, n_init, m_per_year)

# (10, 62, 12, 20) -> (62, 10, 12, 20)
tmp_03 = tmp_03.transpose(1, 0, 2, 3)
tmp_48 = tmp_48.transpose(1, 0, 2, 3)

tmp_03_mean = tmp_03_mean.transpose(1, 0, 2)
tmp_48_mean = tmp_48_mean.transpose(1, 0, 2)

# (62, 10, 12, 20) -> (62, 120, 20)
SPEI_init_03 = tmp_03.reshape(n_init, n_lead * m_per_year, *SPEI_03.shape[2:])
SPEI_init_48 = tmp_48.reshape(n_init, n_lead * m_per_year, *SPEI_48.shape[2:])

SPEI_init_03_mean = tmp_03_mean.reshape(n_init, n_lead * m_per_year)
SPEI_init_48_mean = tmp_48_mean.reshape(n_init, n_lead * m_per_year)

ds_SPEI = xr.Dataset(
    data_vars={
        "SPEI_03": (("init_year", "lead_time_month", "member"), SPEI_init_03),
        "SPEI_48": (("init_year", "lead_time_month", "member"), SPEI_init_48),
        "SPEI_03_ensmean": (("init_year", "lead_time_month",), SPEI_init_03_mean),
        "SPEI_48_ensmean": (("init_year", "lead_time_month",), SPEI_init_48_mean),
    },
    coords={
        "init_year": np.arange(1958+1, 2020+1),
        "lead_time_month": np.arange(120),
        "member": np.arange(11, 31)
    },
)

ds_SPEI_clean = ds_SPEI.where(ds_SPEI >= -4.5)

coarsened = ds_SPEI_clean.coarsen(lead_time_month=12)
ds_SPEI_annual = xr.merge([
    coarsened.min(skipna=True).rename({v: f"{v}_min"  for v in ds_SPEI.data_vars}),
    coarsened.mean(skipna=True).rename({v: f"{v}_mean" for v in ds_SPEI.data_vars}),
]).rename({"lead_time_month": "lead_year"}).assign_coords(lead_year=np.arange(10))

ds_SPEI_annual = ds_SPEI_annual.interpolate_na(dim="init_year", method="linear", max_gap=999)

ds_all = ds_SPEI_annual
ds_all["valid_year"] = ds_all["init_year"] + ds_all["lead_year"]

dir_stn = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{stn}/'
save_name = dir_stn + "CESM_STN.zarr"
ds_all.to_zarr(save_name, mode="w")
print(save_name)


# ============================================================== #
# Yuma PG
#  - PRECT_1d_max       / PRECT_1d_max_ensmean
#  - PRECT_7d_max       / PRECT_7d_max_ensmean
#  - days_above_999p    / days_above_999p_ensmean
#       threshold (clim_p999) is computed from the ensemble-mean
#       PRECT series and applied to both the per-member PRECT
#       and the ensemble-mean PRECT
#  - SPEI
# ============================================================== #

stn = 'Yuma_PG'
vars_keep = ["PRECT",]
years = range(1958, 2020)

ds_collection = []

for year in years:
    fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM_SMYLE_STN_MEMBER/{stn}_{year}.zarr'
    ds = xr.open_zarr(fn, consolidated=True)[vars_keep].chunk({"time": -1})
    ds = ds.sel(time=slice(f'{year+1}-01-01', f'{year+10}-12-31'))

    # ------------------------------------------ #
    ds['PRECT'] = ds['PRECT'] * 60*60*24 * 1000
    # ------------------------------------------ #

    # ---- per-member 1d / 7d max ----
    ds_TP_group = ds.groupby("time.year")
    ds_TP_max  = ds_TP_group.max(dim="time",  skipna=True)
    ds_TP_7d = ds_TP_group.map(
        lambda x: x.rolling(time=7, min_periods=7).mean().max(dim="time", skipna=True)
    )
    ds_TP_max = ds_TP_max.rename({v: f"{v}_1d_max" for v in ds_TP_max.data_vars})
    ds_TP_7d  = ds_TP_7d.rename({v: f"{v}_7d_max"  for v in ds_TP_7d.data_vars})

    # ---- ensemble-mean inputs, then 1d / 7d max ----
    ds_em = ds.mean(dim='member')
    ds_em_TP_group = ds_em.groupby("time.year")
    ds_em_TP_max = ds_em_TP_group.max(dim="time", skipna=True)
    ds_em_TP_7d = ds_em_TP_group.map(
        lambda x: x.rolling(time=7, min_periods=7).mean().max(dim="time", skipna=True)
    )
    ds_em_TP_max = ds_em_TP_max.rename({v: f"{v}_1d_max_ensmean" for v in ds_em_TP_max.data_vars})
    ds_em_TP_7d  = ds_em_TP_7d.rename({v: f"{v}_7d_max_ensmean"  for v in ds_em_TP_7d.data_vars})

    ds_merge = xr.merge([ds_TP_max, ds_TP_7d, ds_em_TP_max, ds_em_TP_7d])
    ds_merge = ds_merge.assign_coords({'year': np.arange(year, year+10) - year})
    ds_collection.append(ds_merge)

ds_TP_max = xr.concat(ds_collection, dim='init_year')
ds_TP_max = ds_TP_max.rename({'year': 'lead_year'})
ds_TP_max = ds_TP_max.assign_coords({'init_year': np.arange(1958+1, 2020+1)})
ds_TP_max = ds_TP_max.chunk({"init_year": -1, "lead_year": -1, "member": -1})
ds_TP_max = ds_TP_max.transpose('init_year', 'lead_year', 'member', missing_dims='ignore')

# ------------------------------------------------------------------ #
# 99.9th-percentile threshold per lead_year, computed from the
# ensemble-mean PRECT series. clim_p999 has only a lead_year dim and
# is applied to both per-member PRECT and ensemble-mean PRECT below.
#
# Note: previously this loop used the stale variable `year` (leftover
# from the loop above) instead of `year_init`, which caused every
# iteration to load stn_2019 and most slices to be empty. Fixed here.
# ------------------------------------------------------------------ #
ds_all_lead_collection = []

for lead_year in range(0, 10):

    start_date = f"{1959+lead_year}-01-01T00"
    end_date = f"{2020+lead_year}-12-31T23"

    ds_collection = []

    for year_init in range(1958, 2020, 1):

        year_start = year_init + 1 + lead_year
        time_start = f'{year_start}-01-01T00'
        time_end = f'{year_start}-12-31T00'

        fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM_SMYLE_STN_MEMBER/{stn}_{year_init}.zarr'
        ds = xr.open_zarr(fn, consolidated=True)[vars_keep].chunk({"time": -1})
        ds = ds.sel(time=slice(f'{year_init+1}-01-01', f'{year_init+10}-12-31'))

        # ------------------------------------------ #
        ds['PRECT'] = ds['PRECT'] * 60*60*24 * 1000
        # ------------------------------------------ #

        # ensemble mean BEFORE taking the quantile, so clim_p999 has no member dim
        ds = ds.mean(dim='member')
        ds = ds.sel(time=slice(time_start, time_end))
        ds_collection.append(ds)

    ds_all = xr.concat(ds_collection, dim='time')
    ds_all_lead_collection.append(ds_all)

ds_all_lead = xr.concat(ds_all_lead_collection, dim='lead_year')
da = ds_all_lead["PRECT"].chunk({"time": -1})
clim_p999 = da.quantile(0.999, dim="time", skipna=True)
clim_p999 = clim_p999.load()  # dims: (lead_year,)

# Apply clim_p999 to both per-member and ensemble-mean PRECT
list_init    = []
list_init_em = []

for year in range(1958, 2020):

    fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM_SMYLE_STN_MEMBER/{stn}_{year}.zarr'
    ds = xr.open_zarr(fn, consolidated=True)[vars_keep].chunk({"time": -1})
    ds = ds.sel(time=slice(f'{year+1}-01-01', f'{year+10}-12-31'))

    # ------------------------------------------ #
    ds['PRECT'] = ds['PRECT'] * 60*60*24 * 1000
    # ------------------------------------------ #

    ds_em = ds.mean(dim='member')

    list_event_collect    = []
    list_event_collect_em = []

    for lead_year in range(10):
        year_now = year + 1 + lead_year
        thres = clim_p999.isel(lead_year=lead_year)  # scalar

        # per-member count vs ensemble-mean threshold
        ds_sub = ds.sel(time=slice(f'{year_now}-01-01T00', f'{year_now}-12-31T00'))
        ds_event_sub = (ds_sub > thres).sum(dim='time')
        list_event_collect.append(ds_event_sub)

        # ensemble-mean count vs same threshold
        ds_em_sub = ds_em.sel(time=slice(f'{year_now}-01-01T00', f'{year_now}-12-31T00'))
        ds_event_sub_em = (ds_em_sub > thres).sum(dim='time')
        list_event_collect_em.append(ds_event_sub_em)

    ds_event    = xr.concat(list_event_collect,    dim='lead_year')
    ds_event_em = xr.concat(list_event_collect_em, dim='lead_year')
    list_init.append(ds_event)
    list_init_em.append(ds_event_em)

ds_event_all = xr.concat(list_init, dim='init_year')
ds_event_all = ds_event_all.assign_coords({'init_year': np.arange(1958+1, 2020+1)})
ds_event_all = ds_event_all.assign_coords({'lead_year': np.arange(10)})
ds_event_all = ds_event_all.rename({'PRECT': 'days_above_999p'})
ds_event_all = ds_event_all.chunk({'init_year': -1, 'lead_year': -1, 'member': -1})

ds_event_all_em = xr.concat(list_init_em, dim='init_year')
ds_event_all_em = ds_event_all_em.assign_coords({'init_year': np.arange(1958+1, 2020+1)})
ds_event_all_em = ds_event_all_em.assign_coords({'lead_year': np.arange(10)})
ds_event_all_em = ds_event_all_em.rename({'PRECT': 'days_above_999p_ensmean'})
ds_event_all_em = ds_event_all_em.chunk({'init_year': -1, 'lead_year': -1})

ds_event_all = xr.merge([ds_event_all, ds_event_all_em])

# SPEI
SPEI_24 = np.empty((10, 12*(2020-1959+1), 20))
SPEI_24[...] = np.nan

SPEI_48 = np.empty((10, 12*(2020-1959+1), 20))
SPEI_48[...] = np.nan

SPEI_24_mean = np.empty((10, 12*(2020-1959+1),))
SPEI_24_mean[...] = np.nan

SPEI_48_mean = np.empty((10, 12*(2020-1959+1),))
SPEI_48_mean[...] = np.nan

for lead_year in range(0, 10):
    print(f'process lead year: {lead_year}')
    # ------------------------------------------------------------ #
    # derive param from ensemble mean
    # ------------------------------------------------------------ #
    start_date = f"{1959+lead_year}-01-01T00"
    end_date = f"{2020+lead_year}-12-31T23"

    ds_collection = []

    for year_init in range(1958, 2020, 1):

        year_start = year_init + 1 + lead_year
        time_start = f'{year_start}-01-01T00'
        time_end = f'{year_start}-12-31T00'

        fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM_SMYLE_STN_MEMBER/{stn}_{year_init}.zarr'
        ds_CESM = xr.open_zarr(fn)[['TREFHTMN', 'TREFHTMX', 'PRECT']].mean(['member',])
        ds_CESM = ds_CESM.sel(time=slice(time_start, time_end))
        ds_collection.append(ds_CESM)

    ds_all = xr.concat(ds_collection, dim='time')
    ds_all = ds_all.load()

    cft = ds_all.indexes['time']
    pd_idx = cft.to_datetimeindex(unsafe=True)
    ds_all = ds_all.assign_coords({'time': pd_idx})

    lat_ref = ds_all['lat'].values
    lat_mid = lat_ref
    time_vals = ds_all['time']

    tmin = ds_all['TREFHTMN'].values
    tmax = ds_all['TREFHTMX'].values
    precip = ds_all['PRECT'].values

    ds = xr.Dataset(
        {
            "precip": (("time",), precip*1e3, {"units": "kg m-2 s-1"}),
            "tmin":   (("time",), tmin-273.15, {"units": "degC"}),
            "tmax":   (("time",), tmax-273.15, {"units": "degC"}),
        },
        coords={"time": time_vals, "lat": lat_mid}
    )

    for v in ("precip", "tmin", "tmax"):
        ds[v] = ds[v].assign_coords(lat=lat_mid)
        ds[v]["lat"].attrs = {
            "standard_name": "latitude",
            "units": "degrees_north", "axis": "Y"
        }

    precip, tmin, tmax = process_vars(ds)

    wb = water_budget(
        pr=precip, tasmin=tmin, tasmax=tmax,
        method="HG85", lat=precip["lat"],
    )
    wb.attrs["units"] = "kg m-2 s-1"

    params_24 = standardized_index_fit_params(
        wb.sel(time=slice(cal_start, cal_end)),
        freq="MS",
        window=24,
        dist=dist,
        method=method,
    )

    params_48 = standardized_index_fit_params(
        wb.sel(time=slice(cal_start, cal_end)),
        freq="MS",
        window=48,
        dist=dist,
        method=method,
    )

    spei_24 = standardized_index(
        wb, freq="MS", window=24, dist=dist, method=method,
        params=params_24, zero_inflated=False, fitkwargs=None,
        cal_start=None, cal_end=None,
    )

    spei_48 = standardized_index(
        wb, freq="MS", window=48, dist=dist, method=method,
        params=params_48, zero_inflated=False, fitkwargs=None,
        cal_start=None, cal_end=None,
    )

    SPEI_24_mean[lead_year, :] = spei_24.values
    SPEI_48_mean[lead_year, :] = spei_48.values

    # ------------------------------------------------------------ #
    # SPEI on members
    # ------------------------------------------------------------ #

    for member in range(20):
        start_date = f"{1959+lead_year}-01-01T00"
        end_date = f"{2020+lead_year}-12-31T23"

        ds_collection = []

        for year_init in range(1958, 2020, 1):

            year_start = year_init + 1 + lead_year
            time_start = f'{year_start}-01-01T00'
            time_end = f'{year_start}-12-31T00'

            fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM_SMYLE_STN_MEMBER/{stn}_{year_init}.zarr'
            ds_CESM = xr.open_zarr(fn)[['TREFHTMN', 'TREFHTMX', 'PRECT']].isel(member=member)
            ds_CESM = ds_CESM.sel(time=slice(time_start, time_end))
            ds_collection.append(ds_CESM)

        ds_all = xr.concat(ds_collection, dim='time')
        ds_all = ds_all.load()

        cft = ds_all.indexes['time']
        pd_idx = cft.to_datetimeindex(unsafe=True)
        ds_all = ds_all.assign_coords({'time': pd_idx})

        lat_ref = ds_all['lat'].values
        lat_mid = lat_ref
        time_vals = ds_all['time']

        tmin = ds_all['TREFHTMN'].values
        tmax = ds_all['TREFHTMX'].values
        precip = ds_all['PRECT'].values

        ds = xr.Dataset(
            {
                "precip": (("time",), precip*1e3, {"units": "kg m-2 s-1"}),
                "tmin":   (("time",), tmin-273.15, {"units": "degC"}),
                "tmax":   (("time",), tmax-273.15, {"units": "degC"}),
            },
            coords={"time": time_vals, "lat": lat_mid}
        )

        for v in ("precip", "tmin", "tmax"):
            ds[v] = ds[v].assign_coords(lat=lat_mid)
            ds[v]["lat"].attrs = {
                "standard_name": "latitude",
                "units": "degrees_north", "axis": "Y"
            }

        precip, tmin, tmax = process_vars(ds)

        wb = water_budget(
            pr=precip, tasmin=tmin, tasmax=tmax,
            method="HG85", lat=precip["lat"],
        )
        wb.attrs["units"] = "kg m-2 s-1"

        spei_24 = standardized_index(
            wb, freq="MS", window=24, dist=dist, method=method,
            params=params_24, zero_inflated=False, fitkwargs=None,
            cal_start=None, cal_end=None,
        )

        spei_48 = standardized_index(
            wb, freq="MS", window=48, dist=dist, method=method,
            params=params_48, zero_inflated=False, fitkwargs=None,
            cal_start=None, cal_end=None,
        )

        SPEI_24[lead_year, :, member] = spei_24.values
        SPEI_48[lead_year, :, member] = spei_48.values


# ======================================= #
# numpy to ds
n_lead = 10
n_init = 62
m_per_year = 12

# (10, 744, 20) -> (10, 62, 12, 20)
tmp_24 = SPEI_24.reshape(n_lead, n_init, m_per_year, *SPEI_24.shape[2:])
tmp_48 = SPEI_48.reshape(n_lead, n_init, m_per_year, *SPEI_48.shape[2:])

tmp_24_mean = SPEI_24_mean.reshape(n_lead, n_init, m_per_year)
tmp_48_mean = SPEI_48_mean.reshape(n_lead, n_init, m_per_year)

# (10, 62, 12, 20) -> (62, 10, 12, 20)
tmp_24 = tmp_24.transpose(1, 0, 2, 3)
tmp_48 = tmp_48.transpose(1, 0, 2, 3)

tmp_24_mean = tmp_24_mean.transpose(1, 0, 2)
tmp_48_mean = tmp_48_mean.transpose(1, 0, 2)

# (62, 10, 12, 20) -> (62, 120, 20)
SPEI_init_24 = tmp_24.reshape(n_init, n_lead * m_per_year, *SPEI_24.shape[2:])
SPEI_init_48 = tmp_48.reshape(n_init, n_lead * m_per_year, *SPEI_48.shape[2:])

SPEI_init_24_mean = tmp_24_mean.reshape(n_init, n_lead * m_per_year)
SPEI_init_48_mean = tmp_48_mean.reshape(n_init, n_lead * m_per_year)

ds_SPEI = xr.Dataset(
    data_vars={
        "SPEI_24": (("init_year", "lead_time_month", "member"), SPEI_init_24),
        "SPEI_48": (("init_year", "lead_time_month", "member"), SPEI_init_48),
        "SPEI_24_ensmean": (("init_year", "lead_time_month",), SPEI_init_24_mean),
        "SPEI_48_ensmean": (("init_year", "lead_time_month",), SPEI_init_48_mean),
    },
    coords={
        "init_year": np.arange(1958+1, 2020+1),
        "lead_time_month": np.arange(120),
        "member": np.arange(11, 31)
    },
)

ds_SPEI_clean = ds_SPEI.where(ds_SPEI >= -4.5)

coarsened = ds_SPEI_clean.coarsen(lead_time_month=12)

ds_SPEI_annual = xr.merge([
    coarsened.min(skipna=True).rename({v: f"{v}_min"  for v in ds_SPEI.data_vars}),
    coarsened.mean(skipna=True).rename({v: f"{v}_mean" for v in ds_SPEI.data_vars}),
]).rename({"lead_time_month": "lead_year"}).assign_coords(lead_year=np.arange(10))

ds_SPEI_annual = ds_SPEI_annual.interpolate_na(dim="init_year", method="linear", max_gap=999)

ds_all = xr.merge([ds_TP_max, ds_event_all, ds_SPEI_annual])
ds_all["valid_year"] = ds_all["init_year"] + ds_all["lead_year"]

dir_stn = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{stn}/'
save_name = dir_stn + "CESM_STN.zarr"
ds_all.to_zarr(save_name, mode="w")
print(save_name)


# ============================================================== #
# Fort Bragg
#  - PRECT_1d_max  / PRECT_1d_max_ensmean
#  - PRECT_3d_max  / PRECT_3d_max_ensmean
#  - PRECT_5d_max  / PRECT_5d_max_ensmean
#  - SPEI
# ============================================================== #

stn = 'Fort_Bragg'
vars_keep = ["PRECT",]
years = range(1958, 2020)

ds_collection = []

for year in range(1958, 2020):

    # get data and variable
    fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM_SMYLE_STN_MEMBER/{stn}_{year}.zarr'
    ds = xr.open_zarr(fn, consolidated=True)[vars_keep].chunk({"time": -1})
    ds = ds.sel(time=slice(f'{year+1}-01-01', f'{year+10}-12-31'))

    # ------------------------------------------ #
    ds['PRECT'] = ds['PRECT'] * 60*60*24 * 1000
    # ------------------------------------------ #

    # ---- per-member 1d / 3d / 5d max ----
    ds_TP_group = ds[["PRECT"]].groupby("time.year")
    ds_TP_max  = ds_TP_group.max(dim="time",  skipna=True)
    ds_TP_3d = ds_TP_group.map(lambda x: x.rolling(time=3, min_periods=3).mean().max(dim="time", skipna=True))
    ds_TP_5d = ds_TP_group.map(lambda x: x.rolling(time=5, min_periods=5).mean().max(dim="time", skipna=True))

    ds_TP_max = ds_TP_max.rename({v: f"{v}_1d_max" for v in ds_TP_max.data_vars})
    ds_TP_3d  = ds_TP_3d.rename({v: f"{v}_3d_max"  for v in ds_TP_3d.data_vars})
    ds_TP_5d  = ds_TP_5d.rename({v: f"{v}_5d_max"  for v in ds_TP_5d.data_vars})

    # ---- ensemble-mean inputs, then 1d / 3d / 5d max ----
    ds_em = ds[["PRECT"]].mean(dim='member')
    ds_em_TP_group = ds_em.groupby("time.year")
    ds_em_TP_max = ds_em_TP_group.max(dim="time", skipna=True)
    ds_em_TP_3d = ds_em_TP_group.map(lambda x: x.rolling(time=3, min_periods=3).mean().max(dim="time", skipna=True))
    ds_em_TP_5d = ds_em_TP_group.map(lambda x: x.rolling(time=5, min_periods=5).mean().max(dim="time", skipna=True))

    ds_em_TP_max = ds_em_TP_max.rename({v: f"{v}_1d_max_ensmean" for v in ds_em_TP_max.data_vars})
    ds_em_TP_3d  = ds_em_TP_3d.rename({v: f"{v}_3d_max_ensmean"  for v in ds_em_TP_3d.data_vars})
    ds_em_TP_5d  = ds_em_TP_5d.rename({v: f"{v}_5d_max_ensmean"  for v in ds_em_TP_5d.data_vars})

    # ============ #
    ds_merge = xr.merge([
        ds_TP_max, ds_TP_3d, ds_TP_5d,
        ds_em_TP_max, ds_em_TP_3d, ds_em_TP_5d,
    ])
    ds_merge = ds_merge.assign_coords({'year': np.arange(year, year+10) - year})
    ds_collection.append(ds_merge)

ds_TP_max = xr.concat(ds_collection, dim='init_year')
ds_TP_max = ds_TP_max.rename({'year': 'lead_year'})
ds_TP_max = ds_TP_max.assign_coords({'init_year': np.arange(1958+1, 2020+1)})
ds_TP_max = ds_TP_max.chunk({"init_year": -1, "lead_year": -1, "member": -1})
ds_TP_max = ds_TP_max.transpose('init_year', 'lead_year', 'member', missing_dims='ignore')


SPEI_09 = np.empty((10, 12*(2020-1959+1), 20))
SPEI_09[...] = np.nan

SPEI_48 = np.empty((10, 12*(2020-1959+1), 20))
SPEI_48[...] = np.nan

SPEI_09_mean = np.empty((10, 12*(2020-1959+1),))
SPEI_09_mean[...] = np.nan

SPEI_48_mean = np.empty((10, 12*(2020-1959+1),))
SPEI_48_mean[...] = np.nan

for lead_year in range(0, 10):
    print(f'process lead year: {lead_year}')
    # ------------------------------------------------------------ #
    # derive param from ensemble mean
    # ------------------------------------------------------------ #
    start_date = f"{1959+lead_year}-01-01T00"
    end_date = f"{2020+lead_year}-12-31T23"

    ds_collection = []

    for year_init in range(1958, 2020, 1):

        year_start = year_init + 1 + lead_year
        time_start = f'{year_start}-01-01T00'
        time_end = f'{year_start}-12-31T00'

        fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM_SMYLE_STN_MEMBER/{stn}_{year_init}.zarr'
        ds_CESM = xr.open_zarr(fn)[['TREFHTMN', 'TREFHTMX', 'PRECT']].mean(['member',])
        ds_CESM = ds_CESM.sel(time=slice(time_start, time_end))
        ds_collection.append(ds_CESM)

    ds_all = xr.concat(ds_collection, dim='time')
    ds_all = ds_all.load()

    cft = ds_all.indexes['time']
    pd_idx = cft.to_datetimeindex(unsafe=True)
    ds_all = ds_all.assign_coords({'time': pd_idx})

    lat_ref = ds_all['lat'].values
    lat_mid = lat_ref
    time_vals = ds_all['time']

    tmin = ds_all['TREFHTMN'].values
    tmax = ds_all['TREFHTMX'].values
    precip = ds_all['PRECT'].values

    ds = xr.Dataset(
        {
            "precip": (("time",), precip*1e3, {"units": "kg m-2 s-1"}),
            "tmin":   (("time",), tmin-273.15, {"units": "degC"}),
            "tmax":   (("time",), tmax-273.15, {"units": "degC"}),
        },
        coords={"time": time_vals, "lat": lat_mid}
    )

    for v in ("precip", "tmin", "tmax"):
        ds[v] = ds[v].assign_coords(lat=lat_mid)
        ds[v]["lat"].attrs = {
            "standard_name": "latitude",
            "units": "degrees_north", "axis": "Y"
        }

    precip, tmin, tmax = process_vars(ds)

    wb = water_budget(
        pr=precip, tasmin=tmin, tasmax=tmax,
        method="HG85", lat=precip["lat"],
    )
    wb.attrs["units"] = "kg m-2 s-1"

    params_09 = standardized_index_fit_params(
        wb.sel(time=slice(cal_start, cal_end)),
        freq="MS",
        window=9,
        dist=dist,
        method=method,
    )

    params_48 = standardized_index_fit_params(
        wb.sel(time=slice(cal_start, cal_end)),
        freq="MS",
        window=48,
        dist=dist,
        method=method,
    )

    spei_09 = standardized_index(
        wb, freq="MS", window=9, dist=dist, method=method,
        params=params_09, zero_inflated=False, fitkwargs=None,
        cal_start=None, cal_end=None,
    )

    spei_48 = standardized_index(
        wb, freq="MS", window=48, dist=dist, method=method,
        params=params_48, zero_inflated=False, fitkwargs=None,
        cal_start=None, cal_end=None,
    )

    SPEI_09_mean[lead_year, :] = spei_09.values
    SPEI_48_mean[lead_year, :] = spei_48.values

    # ------------------------------------------------------------ #
    # SPEI on members
    # ------------------------------------------------------------ #

    for member in range(20):
        start_date = f"{1959+lead_year}-01-01T00"
        end_date = f"{2020+lead_year}-12-31T23"

        ds_collection = []

        for year_init in range(1958, 2020, 1):

            year_start = year_init + 1 + lead_year
            time_start = f'{year_start}-01-01T00'
            time_end = f'{year_start}-12-31T00'

            fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM_SMYLE_STN_MEMBER/{stn}_{year_init}.zarr'
            ds_CESM = xr.open_zarr(fn)[['TREFHTMN', 'TREFHTMX', 'PRECT']].isel(member=member)
            ds_CESM = ds_CESM.sel(time=slice(time_start, time_end))
            ds_collection.append(ds_CESM)

        ds_all = xr.concat(ds_collection, dim='time')
        ds_all = ds_all.load()

        cft = ds_all.indexes['time']
        pd_idx = cft.to_datetimeindex(unsafe=True)
        ds_all = ds_all.assign_coords({'time': pd_idx})

        lat_ref = ds_all['lat'].values
        lat_mid = lat_ref
        time_vals = ds_all['time']

        tmin = ds_all['TREFHTMN'].values
        tmax = ds_all['TREFHTMX'].values
        precip = ds_all['PRECT'].values

        ds = xr.Dataset(
            {
                "precip": (("time",), precip*1e3, {"units": "kg m-2 s-1"}),
                "tmin":   (("time",), tmin-273.15, {"units": "degC"}),
                "tmax":   (("time",), tmax-273.15, {"units": "degC"}),
            },
            coords={"time": time_vals, "lat": lat_mid}
        )

        for v in ("precip", "tmin", "tmax"):
            ds[v] = ds[v].assign_coords(lat=lat_mid)
            ds[v]["lat"].attrs = {
                "standard_name": "latitude",
                "units": "degrees_north", "axis": "Y"
            }

        precip, tmin, tmax = process_vars(ds)

        wb = water_budget(
            pr=precip, tasmin=tmin, tasmax=tmax,
            method="HG85", lat=precip["lat"],
        )
        wb.attrs["units"] = "kg m-2 s-1"

        spei_09 = standardized_index(
            wb, freq="MS", window=9, dist=dist, method=method,
            params=params_09, zero_inflated=False, fitkwargs=None,
            cal_start=None, cal_end=None,
        )

        spei_48 = standardized_index(
            wb, freq="MS", window=48, dist=dist, method=method,
            params=params_48, zero_inflated=False, fitkwargs=None,
            cal_start=None, cal_end=None,
        )

        SPEI_09[lead_year, :, member] = spei_09.values
        SPEI_48[lead_year, :, member] = spei_48.values

# ======================================= #
# numpy to ds
n_lead = 10
n_init = 62
m_per_year = 12

# (10, 744, 20) -> (10, 62, 12, 20)
tmp_09 = SPEI_09.reshape(n_lead, n_init, m_per_year, *SPEI_09.shape[2:])
tmp_48 = SPEI_48.reshape(n_lead, n_init, m_per_year, *SPEI_48.shape[2:])

tmp_09_mean = SPEI_09_mean.reshape(n_lead, n_init, m_per_year)
tmp_48_mean = SPEI_48_mean.reshape(n_lead, n_init, m_per_year)

# (10, 62, 12, 20) -> (62, 10, 12, 20)
tmp_09 = tmp_09.transpose(1, 0, 2, 3)
tmp_48 = tmp_48.transpose(1, 0, 2, 3)

tmp_09_mean = tmp_09_mean.transpose(1, 0, 2)
tmp_48_mean = tmp_48_mean.transpose(1, 0, 2)

# (62, 10, 12, 20) -> (62, 120, 20)
SPEI_init_09 = tmp_09.reshape(n_init, n_lead * m_per_year, *SPEI_09.shape[2:])
SPEI_init_48 = tmp_48.reshape(n_init, n_lead * m_per_year, *SPEI_48.shape[2:])

SPEI_init_09_mean = tmp_09_mean.reshape(n_init, n_lead * m_per_year)
SPEI_init_48_mean = tmp_48_mean.reshape(n_init, n_lead * m_per_year)

ds_SPEI = xr.Dataset(
    data_vars={
        "SPEI_09": (("init_year", "lead_time_month", "member"), SPEI_init_09),
        "SPEI_48": (("init_year", "lead_time_month", "member"), SPEI_init_48),
        "SPEI_09_ensmean": (("init_year", "lead_time_month",), SPEI_init_09_mean),
        "SPEI_48_ensmean": (("init_year", "lead_time_month",), SPEI_init_48_mean),
    },
    coords={
        "init_year": np.arange(1958+1, 2020+1),
        "lead_time_month": np.arange(120),
        "member": np.arange(11, 31)
    },
)

ds_SPEI_clean = ds_SPEI.where(ds_SPEI >= -4.5)

coarsened = ds_SPEI_clean.coarsen(lead_time_month=12)
ds_SPEI_annual = xr.merge([
    coarsened.min(skipna=True).rename({v: f"{v}_min"  for v in ds_SPEI.data_vars}),
    coarsened.mean(skipna=True).rename({v: f"{v}_mean" for v in ds_SPEI.data_vars}),
]).rename({"lead_time_month": "lead_year"}).assign_coords(lead_year=np.arange(10))

ds_SPEI_annual = ds_SPEI_annual.interpolate_na(dim="init_year", method="linear", max_gap=999)

ds_all = xr.merge([ds_TP_max, ds_SPEI_annual])
ds_all["valid_year"] = ds_all["init_year"] + ds_all["lead_year"]

dir_stn = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{stn}/'
save_name = dir_stn + "CESM_STN.zarr"
ds_all.to_zarr(save_name, mode="w")
print(save_name)



# ============================================================== #
# Save all
# ============================================================== #
stn_names = ['Fairbanks', 'Pituffik', 'Guam', 'Yuma_PG', 'Fort_Bragg']

list_ds = []
for stn in stn_names:
    fn = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{stn}/' + "CESM_STN.zarr"
    ds = xr.open_zarr(fn)
    try:
        ds = ds.drop_vars(['lat', 'lon'], errors='ignore')
    except:
        pass
        
    # keep valid_year shared across stations (don't prefix it)
    if "valid_year" in ds.data_vars:
        ds = ds.set_coords("valid_year")

    # prefix every remaining data variable with the station name
    ds = ds.rename({v: f'{stn}_{v}' for v in ds.data_vars})

    list_ds.append(ds)

ds_all = xr.merge(list_ds)

ds_all = ds_all.drop_vars(['quantile',])

save_name = '/glade/derecho/scratch/ksha/EPRI_data/METRICS/STN_CESM_ALL.zarr'
ds_all.to_zarr(save_name, mode='w')
print(save_name)

/glade/derecho/scratch/ksha/EPRI_data/METRICS/Fairbanks/CESM_STN.zarr
/glade/derecho/scratch/ksha/EPRI_data/METRICS/Pituffik/CESM_STN.zarr
process lead year: 0
process lead year: 1
process lead year: 2
process lead year: 3
process lead year: 4
process lead year: 5
process lead year: 6
process lead year: 7
process lead year: 8
process lead year: 9
/glade/derecho/scratch/ksha/EPRI_data/METRICS/Guam/CESM_STN.zarr
process lead year: 0
process lead year: 1
process lead year: 2
process lead year: 3
process lead year: 4
process lead year: 5
process lead year: 6
process lead year: 7
process lead year: 8
process lead year: 9
/glade/derecho/scratch/ksha/EPRI_data/METRICS/Yuma_PG/CESM_STN.zarr
process lead year: 0
process lead year: 1
process lead year: 2
process lead year: 3
process lead year: 4
process lead year: 5
process lead year: 6
process lead year: 7
process lead year: 8
process lead year: 9
/glade/derecho/scratch/ksha/EPRI_data/METRICS/Fort_Bragg/CESM_STN.zarr


In [2]:
xr.open_zarr('/glade/derecho/scratch/ksha/EPRI_data/METRICS/Fort_Bragg/CESM_STN.zarr')

<xarray.Dataset> Size: 579kB
Dimensions:               (init_year: 62, lead_year: 10, member: 20)
Coordinates:
  * init_year             (init_year) int64 496B 1959 1960 1961 ... 2019 2020
    lat                   float64 8B ...
  * lead_year             (lead_year) int64 80B 0 1 2 3 4 5 6 7 8 9
    lon                   float64 8B ...
  * member                (member) int64 160B 11 12 13 14 15 ... 26 27 28 29 30
Data variables: (12/15)
    PRECT_1d_max          (init_year, lead_year, member) float32 50kB dask.array<chunksize=(62, 10, 20), meta=np.ndarray>
    PRECT_1d_max_ensmean  (init_year, lead_year) float32 2kB dask.array<chunksize=(62, 10), meta=np.ndarray>
    PRECT_3d_max          (init_year, lead_year, member) float32 50kB dask.array<chunksize=(62, 10, 20), meta=np.ndarray>
    PRECT_3d_max_ensmean  (init_year, lead_year) float32 2kB dask.array<chunksize=(62, 10), meta=np.ndarray>
    PRECT_5d_max          (init_year, lead_year, member) float32 50kB dask.array<chunksize=(62, 10, 20), meta=np.ndarray>
    PRECT_5d_max_ensmean  (init_year, lead_year) float32 2kB dask.array<chunksize=(62, 10), meta=np.ndarray>
    ...                    ...
    SPEI_09_min           (init_year, lead_year, member) float64 99kB dask.array<chunksize=(62, 10, 20), meta=np.ndarray>
    SPEI_48_ensmean_mean  (init_year, lead_year) float64 5kB dask.array<chunksize=(62, 10), meta=np.ndarray>
    SPEI_48_ensmean_min   (init_year, lead_year) float64 5kB dask.array<chunksize=(62, 10), meta=np.ndarray>
    SPEI_48_mean          (init_year, lead_year, member) float64 99kB dask.array<chunksize=(62, 10, 20), meta=np.ndarray>
    SPEI_48_min           (init_year, lead_year, member) float64 99kB dask.array<chunksize=(62, 10, 20), meta=np.ndarray>
    valid_year            (init_year, lead_year) int64 5kB dask.array<chunksize=(62, 10), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    case:              b.e21.BSMYLE.f09_g17.1958-11.011
    host:              cheyenne3
    initial_file:      b.e21.SMYLE_IC.f09_g17.1958-11.01.cam.i.1958-11-01-000...
    logname:           nanr
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    source:            CAM
    time_period_freq:  day_1
    topography_file:   /glade/p/cesmdata/cseg/inputdata/atm/cam/topo/fv_0.9x1...

In [3]:
# ============================================================== #
# Save all
# ============================================================== #
stn_names = ['Fairbanks', 'Pituffik', 'Guam', 'Yuma_PG', 'Fort_Bragg']

list_ds = []
for stn in stn_names:
    fn = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{stn}/' + "CESM_STN.zarr"
    ds = xr.open_zarr(fn)
    try:
        ds = ds.drop_vars(['lat', 'lon'], errors='ignore')
    except:
        pass
        
    # keep valid_year shared across stations (don't prefix it)
    if "valid_year" in ds.data_vars:
        ds = ds.set_coords("valid_year")

    # prefix every remaining data variable with the station name
    ds = ds.rename({v: f'{stn}_{v}' for v in ds.data_vars})

    list_ds.append(ds)

ds_all = xr.merge(list_ds)

ds_all = ds_all.drop_vars(['quantile',])

save_name = '/glade/derecho/scratch/ksha/EPRI_data/METRICS/STN_CESM_ALL.zarr'
# ds_all.to_zarr(save_name, mode='w')
print(save_name)

/glade/derecho/scratch/ksha/EPRI_data/METRICS/STN_CESM_ALL.zarr
